# Cascade. Does the grader use the masks?

The gate `F_gated = F_img + gamma * W_o A` starts closed (`gamma = 0`), so the grader could
ignore the masks entirely. Two tests:

1. **No retraining.** Read `gamma` from each fold model, then re-score the 3,714 held-out images
   of fold 0 after removing all masks, after giving every image the masks of another image in
   the batch (in-distribution but wrong; stricter than zeroing), and after removing one channel
   at a time.
2. **Gamma-fix retrain.** `gamma` started at zero under the same weight decay (0.05) as the
   other head parameters. Fold 0 is retrained with `gamma` initialised to 0.1 and excluded
   from weight decay, everything else unchanged, and the ablation is repeated.

All decodes here are the default 0.5 decode (`lgdrg_predict`); QWK uses the paper definition.

## 0. Configuration

In [ ]:
from pathlib import Path

# Development pool (18,570 DR-positive images, grades 1-5) and its Stage II masks.
#   data/stage3/dev_grades.csv              columns: image, grade (1..5)
#   data/stage3/dev_images/<stem>.png|jpg
#   data/stage3/dev_masks/<lesion>/<stem><suffix>.png   lesion in {ma, he, ex, cws, vessel};
#       suffix: ma "_mask", he "_he_mask", ex "_ex_mask", cws "_cws_mask", vessel "_mask"
DATA_ROOT = Path("data")
DEV_CSV = DATA_ROOT / "stage3" / "dev_grades.csv"
DEV_IMG_DIR = DATA_ROOT / "stage3" / "dev_images"
DEV_MASK_ROOT = DATA_ROOT / "stage3" / "dev_masks"

OUT_ROOT = Path("outputs")
CKPT_DIR = OUT_ROOT / "stage3" / "checkpoints"      # best_fold{0..4}.pt
EVAL_DIR = OUT_ROOT / "stage3" / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

import json
import pandas as pd
import torch
from hierarchiretina.stage3.engine import LGDRGConfig
from hierarchiretina.stage3.data import prepare_development_set, grad_pos_weight, make_loaders_2h

cfg = LGDRGConfig(out_dir=str(CKPT_DIR))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FOLD = 0
ABL_DIR = OUT_ROOT / "cascade" / "ablation"
ABL_DIR.mkdir(parents=True, exist_ok=True)

df = prepare_development_set(DEV_CSV, DEV_IMG_DIR, DEV_MASK_ROOT, cfg.n_folds, cfg.seed)
POS_WEIGHT = grad_pos_weight(df)

## 1. Learned gate scale per fold (EMA weights)

In [ ]:
from hierarchiretina.stage3.ablation import read_gamma

gam = read_gamma(CKPT_DIR, cfg.n_folds)
gam.to_csv(ABL_DIR / "gamma_per_fold.csv", index=False)
print(gam.round(4).to_string(index=False))
print(f"mean |gamma| = {gam.gamma.abs().mean():.4f}")

## 2. Mask ablation on fold 0 (original model, no retraining)

In [ ]:
from hierarchiretina.stage3.model import load_lgdrg
from hierarchiretina.stage3.ablation import run_mask_ablation

_, vl, va = make_loaders_2h(df, FOLD, cfg.img_size, cfg.batch_size, cfg.num_workers)
print(f"fold {FOLD} held-out images: {len(va):,}")
model = load_lgdrg(CKPT_DIR / f"best_fold{FOLD}.pt", cfg, DEVICE)
abl_orig = run_mask_ablation(model, vl, DEVICE, cfg.use_amp)
abl_orig.to_csv(ABL_DIR / "ablation_masks.csv", index=False)
del model
abl_orig.round(4)

## 3. Gamma-fix retrain of fold 0

Exactly one change: `gamma` is initialised to 0.1 (before the EMA copy is made) and placed in
its own parameter group with the head learning rate and no weight decay. Data, fold, sampler,
schedule, EMA, epochs, patience and model selection are unchanged; as in the original run,
only `torch` and `numpy` are re-seeded. Checkpoints go to `<checkpoints>_gammafix/`. One fold
takes about as long as one original fold (about 65 min per epoch on an RTX A6000).

In [ ]:
from hierarchiretina.stage3.ablation import gamma_fix_config
from hierarchiretina.stage3.engine import train_fold

gcfg = gamma_fix_config(cfg)
best_auc, gfix_ckpt = train_fold(gcfg, df, FOLD, POS_WEIGHT, device=DEVICE, seed_all=False)

## 4. Mask ablation on the retrained model

In [ ]:
from hierarchiretina.stage3.ablation import summarise_gamma_fix

model = load_lgdrg(gfix_ckpt, gcfg, DEVICE)
abl_fix = run_mask_ablation(model, vl, DEVICE, gcfg.use_amp)
abl_fix.to_csv(ABL_DIR / "gammafix_ablation_masks.csv", index=False)
summary = summarise_gamma_fix(gfix_ckpt, abl_fix)
json.dump(summary, open(ABL_DIR / "gammafix_fold0_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))
del model

## 5. Table: mask reliance of LG-DRG (fold 0 held-out images)

In [ ]:
from hierarchiretina.stage3.ablation import ablation_table

tab = ablation_table(abl_orig, abl_fix)
print(tab.round(4).to_string(index=False))
print(f"\nmacro-AUC, intact: original {abl_orig.macro_auc.iloc[0]:.4f} | retrained {abl_fix.macro_auc.iloc[0]:.4f}")
print(f"learned gamma: original mean |gamma| {gam.gamma.abs().mean():.3f} (all folds) | "
      f"retrained {summary['gamma_best_ema']:.3f}")